<a href="https://colab.research.google.com/github/DenisBoytsov41/tutors-sentiment-coursework/blob/main/notebooks/00_project_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00_project_setup

Подготовка окружения проекта:
- подключение Google Drive;
- проверка структуры папок проекта;
- установка библиотек;
- получение локального доступа к git-репозиторию;
- настройка путей для дальнейших ноутбуков.

Рабочая логика проекта:
- **GitHub** — код и ноутбуки;
- **Google Drive** — данные, модели и результаты;
- **Colab** — среда выполнения.

In [1]:
!pip -q install datasets transformers accelerate evaluate scikit-learn pandas numpy matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import sys
import json
import getpass
import subprocess
from pathlib import Path
from datetime import datetime

In [4]:
# диск
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/tutors_sentiment_project")

ADMIN_DIR = DRIVE_PROJECT_ROOT / "00_admin"
RAW_DIR = DRIVE_PROJECT_ROOT / "01_data_raw"
INTERIM_DIR = DRIVE_PROJECT_ROOT / "02_data_interim"
LABELING_DIR = DRIVE_PROJECT_ROOT / "03_labeling"
READY_DIR = DRIVE_PROJECT_ROOT / "04_datasets_ready"
MODELS_DIR = DRIVE_PROJECT_ROOT / "05_models"
REPORTS_DIR = DRIVE_PROJECT_ROOT / "06_reports"
EXPORTS_DIR = DRIVE_PROJECT_ROOT / "07_exports"
ARCHIVE_DIR = DRIVE_PROJECT_ROOT / "08_archive"

# рабочая папка Colab
LOCAL_REPO_ROOT = Path("/content/tutors-sentiment-coursework")

print("DRIVE_PROJECT_ROOT =", DRIVE_PROJECT_ROOT)
print("LOCAL_REPO_ROOT    =", LOCAL_REPO_ROOT)

DRIVE_PROJECT_ROOT = /content/drive/MyDrive/tutors_sentiment_project
LOCAL_REPO_ROOT    = /content/tutors-sentiment-coursework


In [6]:
required_dirs = [
    ADMIN_DIR,
    RAW_DIR,
    INTERIM_DIR,
    LABELING_DIR,
    READY_DIR,
    MODELS_DIR,
    REPORTS_DIR,
    EXPORTS_DIR,
    ARCHIVE_DIR,
    RAW_DIR / "rusentiment",
    RAW_DIR / "eedi_tutoring",
    RAW_DIR / "esconv",
    RAW_DIR / "student_feedback",
    RAW_DIR / "project_team_analog",
]

for d in required_dirs:
    d.mkdir(parents=True, exist_ok=True)

print("Папки проекта на Google Drive готовы.")

Папки проекта на Google Drive готовы.


In [7]:
# локальный доступ к git-репозиторию

REPO_URL = "https://github.com/DenisBoytsov41/tutors-sentiment-coursework.git"

if LOCAL_REPO_ROOT.exists() and (LOCAL_REPO_ROOT / ".git").exists():
    print("Локальный репозиторий уже существует:", LOCAL_REPO_ROOT)
else:
    print("Локальный репозиторий не найден. Нужно выполнить клонирование.")

    github_username = input("GitHub username: ").strip()
    github_token = getpass.getpass("GitHub token (PAT): ").strip()

    askpass_path = Path("/content/git_askpass_clone.sh")
    askpass_script = """#!/bin/sh
case "$1" in
  *Username*) echo "$GITHUB_USERNAME" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
  *) echo "" ;;
esac
"""
    askpass_path.write_text(askpass_script, encoding="utf-8")
    askpass_path.chmod(0o700)

    env = os.environ.copy()
    env["GITHUB_USERNAME"] = github_username
    env["GITHUB_TOKEN"] = github_token
    env["GIT_ASKPASS"] = str(askpass_path)
    env["GIT_TERMINAL_PROMPT"] = "0"

    subprocess.run(
        ["git", "clone", REPO_URL, str(LOCAL_REPO_ROOT)],
        check=True,
        env=env
    )

    askpass_path.unlink(missing_ok=True)
    env.pop("GITHUB_TOKEN", None)

    print("Репозиторий успешно клонирован:", LOCAL_REPO_ROOT)

Локальный репозиторий не найден. Нужно выполнить клонирование.
GitHub username:  DenisBoytsov41
GitHub token (PAT): ··········
Репозиторий успешно клонирован: /content/tutors-sentiment-coursework


In [8]:
# подключение локального репозитория к Python
if str(LOCAL_REPO_ROOT) not in sys.path:
    sys.path.append(str(LOCAL_REPO_ROOT))

print("Путь добавлен в sys.path")

Путь добавлен в sys.path


In [9]:
# загрузка конфигов проекта

CONFIGS_DIR = LOCAL_REPO_ROOT / "configs"

with open(CONFIGS_DIR / "model_config.json", "r", encoding="utf-8") as f:
    model_config = json.load(f)

with open(CONFIGS_DIR / "data_config.json", "r", encoding="utf-8") as f:
    data_config = json.load(f)

with open(CONFIGS_DIR / "train_config.json", "r", encoding="utf-8") as f:
    train_config = json.load(f)

print("model_config =", model_config)
print("data_config  =", data_config)
print("train_config =", train_config)

model_config = {'base_model': 'blanchefort/rubert-base-cased-sentiment-rusentiment', 'num_labels': 3}
data_config  = {'raw_data_root': '/content/drive/MyDrive/tutors_sentiment_project/01_data_raw', 'interim_data_root': '/content/drive/MyDrive/tutors_sentiment_project/02_data_interim', 'labeling_root': '/content/drive/MyDrive/tutors_sentiment_project/03_labeling', 'ready_data_root': '/content/drive/MyDrive/tutors_sentiment_project/04_datasets_ready'}
train_config = {'epochs': 3, 'batch_size': 16, 'learning_rate': 2e-05, 'max_length': 256}


In [10]:
raw_folders = [
    RAW_DIR / "rusentiment",
    RAW_DIR / "eedi_tutoring",
    RAW_DIR / "esconv",
    RAW_DIR / "student_feedback",
    RAW_DIR / "project_team_analog",
]

for folder in raw_folders:
    files = list(folder.rglob("*"))
    real_files = [p for p in files if p.is_file()]
    print(f"{folder.name}: {len(real_files)} файлов")

rusentiment: 0 файлов
eedi_tutoring: 0 файлов
esconv: 0 файлов
student_feedback: 0 файлов
project_team_analog: 0 файлов


In [11]:
setup_report = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "drive_project_root": str(DRIVE_PROJECT_ROOT),
    "local_repo_root": str(LOCAL_REPO_ROOT),
    "raw_data_folders": {
        "rusentiment": str(RAW_DIR / "rusentiment"),
        "eedi_tutoring": str(RAW_DIR / "eedi_tutoring"),
        "esconv": str(RAW_DIR / "esconv"),
        "student_feedback": str(RAW_DIR / "student_feedback"),
        "project_team_analog": str(RAW_DIR / "project_team_analog"),
    },
    "base_model": model_config.get("base_model"),
    "num_labels": model_config.get("num_labels"),
    "train_config": train_config,
}

setup_report_path = ADMIN_DIR / "setup_report.json"
with open(setup_report_path, "w", encoding="utf-8") as f:
    json.dump(setup_report, f, ensure_ascii=False, indent=2)

print("Отчёт сохранён:", setup_report_path)

Отчёт сохранён: /content/drive/MyDrive/tutors_sentiment_project/00_admin/setup_report.json


In [12]:
print("=" * 60)
print("Проект готов к работе")
print("=" * 60)
print("GitHub repo local :", LOCAL_REPO_ROOT)
print("Google Drive root :", DRIVE_PROJECT_ROOT)
print("Base model        :", model_config.get("base_model"))
print("Num labels        :", model_config.get("num_labels"))
print("Следующий шаг     : notebooks/01_data_inventory.ipynb")

Проект готов к работе
GitHub repo local : /content/tutors-sentiment-coursework
Google Drive root : /content/drive/MyDrive/tutors_sentiment_project
Base model        : blanchefort/rubert-base-cased-sentiment-rusentiment
Num labels        : 3
Следующий шаг     : notebooks/01_data_inventory.ipynb
